In [165]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from datetime import timedelta
import scipy
from math import factorial, exp


In [166]:
CONFIG = {"data_dir": "../data", 
          "data_files": ["prem_21.csv", "prem_22.csv", "prem_23.csv", "prem_24.csv", "prem_25.csv", "prem_26.csv"], 
          "seasons": ["20/21", "21/22", "22/23", "23/24", "24/25", "25/26"], 
          "split": {"method": "chronological_holdout", "valid_seasons": 2, "test_seasons": 1}, 
          "columns": {"date": "Date", 
                      "home_team": "HomeTeam", 
                      "away_team": "AwayTeam", 
                      "home_goals": "FTHG", 
                      "away_goals": "FTAG", 
                      "result": "FTR", 
                      "home_odds": "PSH", 
                      "draw_odds": "PSD", 
                      "away_odds": "PSA"}}

In [167]:
def check_assumptions(config):
    # I understand that it is common practice for strong programmers to baby-proof their work in the interest of both themselves and others,
    # I will try to emulate this

    """
    Assumptions:
        • Correct number of data files
        • Non-empty validation and test set
        • All columns exist
        • File paths exist
    """

    if len(config["data_files"]) != len(config["seasons"]):
        return "No go broski, missing seasons or corresponding data files"
    
    if config["split"]["valid_seasons"] == 0:
        return "Missing validation set"
    
    if config["split"]["test_seasons"] == 0:
        return "Missing test set"
    
    data_dir = Path(config["data_dir"])

    def paths_exist(config):
        for i in range(len(config["data_files"])):
            file_name = config["data_files"][i]
            path = data_dir / file_name
            if not path.exists():
                return f"{i}th path does not exist"
        return True
    
    def columns_exist(config):
        for column in list(config["columns"].keys()):
            value = config["columns"][column]

            if value == None or value.strip() == "":
                return f"{column} is empty"

            for i in range(len(config["data_files"])):
                file_name = config["data_files"][i]
                path = data_dir / file_name

                df = pd.read_csv(path)

                if value not in df.columns:
                    return f'{config["data_files"][i]} is missing column {value} for config key {column}'
                
        return True

    path_check = paths_exist(config)

    if path_check != True:
        return path_check

    column_check = columns_exist(config)

    if column_check != True:
        return column_check

    return "Good to go"
  


In [168]:
def make_training_data(config):
    data_dir = Path(config["data_dir"])
    training_dataframes = []

    # Loop through, make all of the paths, and corresponding dataframes with seasons column and store to be concatenated
    for i in range(len(config["data_files"]) - config["split"]["valid_seasons"] - config["split"]["test_seasons"]):
        file_name = config["data_files"][i]
        season = config["seasons"][i]

        path = data_dir / file_name
        df_unfiltered = pd.read_csv(path)
        df = df_unfiltered.loc[:, [config["columns"]["home_team"], config["columns"]["away_team"], 
                           config["columns"]["home_goals"], config["columns"]["away_goals"], 
                           config["columns"]["result"], config["columns"]["home_odds"], 
                           config["columns"]["draw_odds"], config["columns"]["away_odds"], config["columns"]["date"]]] 

        df["Season"] = season

        training_dataframes.insert(0, df)

    return pd.concat(training_dataframes, ignore_index = True)


In [169]:
df_train = make_training_data(CONFIG)

In [170]:
#The first key correction of dixon coles over maher was the correlation at low scoring games, using law of total prob can manually estimate what these corrections must be

def correlation_function(home_goals, away_goals, lambda_, mu, rho = 0.01):

        # Only 5 cases so we can do it manually as oppsed to solving LOTP equations numerically
        if home_goals == 0 and away_goals == 0:
            multiplyer = 1 - lambda_ * mu * rho
        
        elif home_goals == 0 and away_goals == 1:
            multiplyer = 1 + lambda_ * rho

        elif home_goals == 1 and away_goals == 0:
            multiplyer = 1 + mu * rho

        elif home_goals == 1 and away_goals == 1:
            multiplyer = 1 - rho

        else:
            multiplyer = 1
        
        return multiplyer

In [171]:
def atk_def_coefficients(df_train, config, phi = -np.log(0.9)/365.25):

    home_teams = sorted(list(dict.fromkeys(df_train[config["columns"]["home_team"]])))
    away_teams = sorted(list(dict.fromkeys(df_train[config["columns"]["away_team"]])))

    if home_teams == away_teams:
        teams = home_teams

    else:
        return "Home and Away teams don't match"
    
    teams_dict = {team: i for i, team in enumerate(teams)}

    home_goals_all = []
    away_goals_all = []
    home_team_indices_all = []
    away_team_indices_all = []
    decay_weights_all = []

    dates = pd.to_datetime(df_train[config["columns"]["date"]], format="%d/%m/%Y")
    most_recent_date = dates.max()

    for m in range(len(df_train)):
        home_goals_all.append(df_train[config["columns"]["home_goals"]].iloc[m])
        away_goals_all.append(df_train[config["columns"]["away_goals"]].iloc[m])

        home_team = df_train[config["columns"]["home_team"]].iloc[m]
        away_team = df_train[config["columns"]["away_team"]].iloc[m]

        home_team_indices_all.append(teams_dict.get(home_team))
        away_team_indices_all.append(teams_dict.get(away_team))

        t = (most_recent_date - dates.iloc[m]).days
        decay_weights_all.append(np.exp(-phi * t))


    def likelihood_function(params, config, df_train, phi, rho=0.01):

        l = 0

        for match_index in range(df_train.shape[0]):

            home_goals = home_goals_all[match_index]
            away_goals = away_goals_all[match_index]

            home_idx = home_team_indices_all[match_index]
            away_idx = away_team_indices_all[match_index]

            home_atk = params[home_idx]
            away_def = params[len(teams) + away_idx]

            away_atk = params[away_idx]
            home_def = params[len(teams) + home_idx]

            gamma = params[-1]
            lambda_ = home_atk * away_def * gamma
            mu = home_def * away_atk

            tau = correlation_function(home_goals, away_goals, lambda_, mu, rho)

            l += (np.log(tau) + home_goals * np.log(lambda_) + away_goals * np.log(mu) - lambda_ - mu) * decay_weights_all[match_index]

        return -l
    

    def norm_constraint_function(params):
        atk_coeff_sum = 0
        for i in range(len(teams)):
            atk_coeff_sum += params[i]
        
        return atk_coeff_sum - len(teams)


    initial_params = np.ones(len(teams) * 2 + 1)


    optimal_coeffs = scipy.optimize.minimize(likelihood_function, initial_params, args = (config, df_train, phi), bounds = [(1e-6, 2)] * len(initial_params), constraints = {"type": "eq", "fun": norm_constraint_function})

    coeffs = optimal_coeffs.x
    gamma = coeffs[-1]


    att_def_coeffs_dict = {"Team": [team for team in teams], "AttackCoefficient": [coeffs[i] for i in range(len(teams))], "DefenceCoefficient": [coeffs[i+len(teams)] for i in range(len(teams))]}
    df_atk_def_coeffs = pd.DataFrame(att_def_coeffs_dict)

    print(optimal_coeffs.success)
    print(optimal_coeffs.message)



    return df_atk_def_coeffs, float(gamma)



In [172]:
atk_def_coeffs = atk_def_coefficients(df_train, CONFIG, phi = -np.log(0.85)/365.25)

True
Optimization terminated successfully


In [173]:
def hda_odds(config, df_fixtures, atk_def_coeffs, gamma, rho=0.01,
             team_col="Team", atk_coeff="AttackCoefficient", def_coeff="DefenceCoefficient"):

    home_odds = []
    draw_odds = []
    away_odds = []

    max_goals = 10

    for match_idx in range(df_fixtures.shape[0]):

        score_matrix = []

        home_team = df_fixtures[config["columns"]["home_team"]].iloc[match_idx]
        away_team = df_fixtures[config["columns"]["away_team"]].iloc[match_idx]

        h_atk = atk_def_coeffs.loc[atk_def_coeffs[team_col] == home_team, atk_coeff].iloc[0]
        h_def = atk_def_coeffs.loc[atk_def_coeffs[team_col] == home_team, def_coeff].iloc[0]

        a_atk = atk_def_coeffs.loc[atk_def_coeffs[team_col] == away_team, atk_coeff].iloc[0]
        a_def = atk_def_coeffs.loc[atk_def_coeffs[team_col] == away_team, def_coeff].iloc[0]

        lambda_ = h_atk * a_def * gamma
        mu = h_def * a_atk

        for x in range(max_goals + 1):
            row = []

            for y in range(max_goals + 1):
                tau = correlation_function(x, y, lambda_, mu, rho)
                p = tau * (lambda_ ** x) * (mu ** y) * np.exp(-(lambda_ + mu)) / (factorial(x) * factorial(y))
                row.append(p)

            score_matrix.append(row)

        score_matrix = np.array(score_matrix)

        home_prob = np.tril(score_matrix, k=-1).sum()
        draw_prob = np.trace(score_matrix)
        away_prob = np.triu(score_matrix, k=1).sum()

        total = home_prob + draw_prob + away_prob

        home_prob = home_prob / total
        draw_prob = draw_prob / total
        away_prob = away_prob / total

        home_odds.append(1 / home_prob)
        draw_odds.append(1 / draw_prob)
        away_odds.append(1 / away_prob)

    return pd.DataFrame({config["columns"]["date"]: df_fixtures[config["columns"]["date"]], config["columns"]["home_team"]: df_fixtures[config["columns"]["home_team"]],
        config["columns"]["away_team"]: df_fixtures[config["columns"]["away_team"]], config["columns"]["home_odds"]: home_odds,
        config["columns"]["draw_odds"]: draw_odds, config["columns"]["away_odds"]: away_odds})

In [174]:
def make_validation_data(config, df_fixtures):
    validation_sets = []
    data_dir = Path(config["data_dir"])

    if config["split"]["method"] != "chronological_holdout":
        return ValueError("Unkown split method")
    else:
        for i in range(config["split"]["valid_seasons"]):
            file_name = config["data_files"][len(config["data_files"]) + i - config["split"]["valid_seasons"] - config["split"]["test_seasons"]]
            path = data_dir / file_name
            df_unfiltered = pd.read_csv(path)

            df =  df_unfiltered.loc[:, [config["columns"]["home_team"], config["columns"]["away_team"], 
                           config["columns"]["home_goals"], config["columns"]["away_goals"], 
                           config["columns"]["result"], config["columns"]["home_odds"], 
                           config["columns"]["draw_odds"], config["columns"]["away_odds"], config["columns"]["date"]]] 
            
            validation_sets.insert(0, df)

        df_valid_all = pd.concat(validation_sets, ignore_index=True)

        home_col = config["columns"]["home_team"]
        away_col = config["columns"]["away_team"]

        training_teams = set(df_train[home_col]) | set(df_train[away_col])

        df_valid = df_valid_all[df_valid_all[home_col].isin(training_teams) & df_valid_all[away_col].isin(training_teams)].reset_index(drop=True)

        return df_valid


In [175]:
df_valid = make_validation_data(CONFIG, df_train)

In [176]:
df_hda_odds = hda_odds(CONFIG, df_valid, atk_def_coeffs[0], atk_def_coeffs[1])
print(df_hda_odds.head(10))

         Date       HomeTeam        AwayTeam       PSH       PSD       PSA
0  16/08/2024     Man United          Fulham  1.799406  4.240937  4.796986
1  17/08/2024        Arsenal          Wolves  1.521274  4.754902  7.555896
2  17/08/2024        Everton        Brighton  3.462389  3.727444  2.257837
3  17/08/2024      Newcastle     Southampton  1.625429  4.801164  5.665887
4  17/08/2024  Nott'm Forest     Bournemouth  2.257793  3.860129  3.355357
5  17/08/2024       West Ham     Aston Villa  2.445533  3.860610  3.011461
6  18/08/2024      Brentford  Crystal Palace  2.043338  3.865951  3.969260
7  18/08/2024        Chelsea        Man City  4.606448  4.091341  1.857030
8  19/08/2024      Leicester       Tottenham  2.942729  4.592902  2.260131
9  24/08/2024       Brighton      Man United  2.642978  3.921218  2.727648


In [177]:
def log_loss(config, df_market, df_hda_odds):
    log_loss = 0
    n_matches = df_market.shape[0]

    for match_idx in range(df_market.shape[0]):
        row = df_hda_odds[(df_hda_odds[config["columns"]["home_team"]] == df_market[config["columns"]["home_team"]].iloc[match_idx]) & 
                          (df_hda_odds[config["columns"]["away_team"]] == df_market[config["columns"]["away_team"]].iloc[match_idx])]
        
        home_odds = row[config["columns"]["home_odds"]].iloc[0]
        draw_odds = row[config["columns"]["draw_odds"]].iloc[0]
        away_odds = row[config["columns"]["away_odds"]].iloc[0]

        match_result = df_market[config["columns"]["result"]].iloc[match_idx]

        if match_result == "H":
            log_loss += np.log(home_odds)

        elif match_result == "D":
            log_loss += np.log(draw_odds)

        elif match_result == "A":
            log_loss += np.log(away_odds)

        else:
            raise ValueError("Unkown result")
    
    return log_loss / n_matches

In [178]:
log_loss(CONFIG, df_valid, df_hda_odds)

np.float64(0.9935281381768417)

In [179]:
def decay_corr_optimiser(config, df_fixtures, max_rho = 0.1, min_rho = -0.1, max_decay = 0.999, min_decay = 0, grid_size = 10):

    log_losses = {}

    rhos = np.linspace(min_rho, max_rho, grid_size)
    decays = np.linspace(min_decay, max_decay, grid_size)

    for rho in rhos:
        for decay in decays:

            atk_def_coeffs = atk_def_coefficients(df_train, config, -np.log(1 - decay)/365.25)
            df_hda_odds = hda_odds(config, df_fixtures, atk_def_coeffs[0], atk_def_coeffs[1], rho)

            log_losses[(rho,decay)] = log_loss(config, df_fixtures, df_hda_odds)

    best_params = min(log_losses, key = log_losses.get)
    best_loss = log_losses[best_params]

    return best_params, best_loss




In [180]:
optimised_validation = decay_corr_optimiser(CONFIG, df_valid)

True
Optimization terminated successfully
True
Optimization terminated successfully
True
Optimization terminated successfully
True
Optimization terminated successfully
True
Optimization terminated successfully
True
Optimization terminated successfully
True
Optimization terminated successfully
True
Optimization terminated successfully
True
Optimization terminated successfully
True
Optimization terminated successfully
True
Optimization terminated successfully
True
Optimization terminated successfully
True
Optimization terminated successfully
True
Optimization terminated successfully
True
Optimization terminated successfully
True
Optimization terminated successfully
True
Optimization terminated successfully
True
Optimization terminated successfully
True
Optimization terminated successfully
True
Optimization terminated successfully
True
Optimization terminated successfully
True
Optimization terminated successfully
True
Optimization terminated successfully
True
Optimization terminated succe

In [181]:
print(optimised_validation)

((np.float64(0.033333333333333326), np.float64(0.555)), np.float64(0.9876002423262558))


In [182]:
def brier_score(config, df_fixtures, decay, rho, outcomes = ["H", "D", "A"]):
    brier_scores = []
    atk_def_coeffs = atk_def_coefficients(df_train, config, -np.log(1 - float(decay))/365.25)
    df_hda_odds = hda_odds(config, df_fixtures, atk_def_coeffs[0], atk_def_coeffs[1], float(rho))
    
    for m in range(df_fixtures.shape[0]):
        home_odds = df_hda_odds[config["columns"]["home_odds"]].iloc[m]
        draw_odds = df_hda_odds[config["columns"]["draw_odds"]].iloc[m]
        away_odds = df_hda_odds[config["columns"]["away_odds"]].iloc[m]

        prob_vector = [1/home_odds, 1/draw_odds, 1/away_odds]
        result_vector = [int(df_fixtures[config["columns"]["result"]].iloc[m] == result) for result in outcomes]

        brier_scores.append(np.linalg.norm(np.array(prob_vector) - np.array(result_vector))**2)

    return np.mean(np.array(brier_scores))

In [183]:
brier_score(CONFIG, df_valid, optimised_validation[0][1], optimised_validation[0][0])

True
Optimization terminated successfully


np.float64(0.5889044512557772)

In [207]:
def betting_strategy_a(config, df_fixtures, df_hda_odds, threshold, outcomes = ["H", "D", "A"]):
    returns = 0

    for match_idx in range(df_fixtures.shape[0]):
        home_odds = df_fixtures[config["columns"]["home_odds"]].iloc[match_idx]
        draw_odds = df_fixtures[config["columns"]["draw_odds"]].iloc[match_idx]
        away_odds = df_fixtures[config["columns"]["away_odds"]].iloc[match_idx]

        home_model_prob = 1/df_hda_odds[config["columns"]["home_odds"]].iloc[match_idx]
        draw_model_prob = 1/df_hda_odds[config["columns"]["draw_odds"]].iloc[match_idx]
        away_model_prob = 1/df_hda_odds[config["columns"]["away_odds"]].iloc[match_idx]

        if home_odds * home_model_prob - 1 > threshold:
            returns += home_odds * int(df_fixtures[config["columns"]["result"]].iloc[match_idx] == outcomes[0]) - 1

        if draw_odds * draw_model_prob - 1 > threshold:
            returns += draw_odds * int(df_fixtures[config["columns"]["result"]].iloc[match_idx] == outcomes[1]) - 1

        if away_odds * away_model_prob - 1 > threshold:
            returns += away_odds * int(df_fixtures[config["columns"]["result"]].iloc[match_idx] == outcomes[2]) - 1
        
    return returns

In [187]:
def betting_strategy_b(config, df_fixtures, df_hda_odds, threshold, outcomes = ["H", "D", "A"]):
    returns = 0

    for match_idx in range(df_fixtures.shape[0]):
        home_odds = df_fixtures[config["columns"]["home_odds"]].iloc[match_idx]
        draw_odds = df_fixtures[config["columns"]["draw_odds"]].iloc[match_idx]
        away_odds = df_fixtures[config["columns"]["away_odds"]].iloc[match_idx]

        home_model_prob = 1/df_hda_odds[config["columns"]["home_odds"]].iloc[match_idx]
        draw_model_prob = 1/df_hda_odds[config["columns"]["draw_odds"]].iloc[match_idx]
        away_model_prob = 1/df_hda_odds[config["columns"]["away_odds"]].iloc[match_idx]

        if home_odds * home_model_prob - 1 > 0 and home_odds * home_model_prob - 1 < threshold:
            returns += (home_odds * int(df_fixtures[config["columns"]["result"]].iloc[match_idx] == outcomes[0]) - 1)
        elif home_odds * home_model_prob - 1 > threshold:
            returns += (home_odds * int(df_fixtures[config["columns"]["result"]].iloc[match_idx] == outcomes[0]) - 1) * home_odds * home_model_prob

        if draw_odds * draw_model_prob - 1 > 0 and draw_odds * draw_model_prob - 1 < threshold:
            returns += (draw_odds * int(df_fixtures[config["columns"]["result"]].iloc[match_idx] == outcomes[1]) - 1)
        elif draw_odds * draw_model_prob - 1 > threshold:
            returns += (draw_odds * int(df_fixtures[config["columns"]["result"]].iloc[match_idx] == outcomes[1]) - 1) * draw_odds * draw_model_prob

        if away_odds * away_model_prob - 1 > 0 and away_odds * away_model_prob - 1 < threshold:
            returns += (away_odds * int(df_fixtures[config["columns"]["result"]].iloc[match_idx] == outcomes[2]) - 1)
        elif away_odds * away_model_prob - 1 > threshold:
            returns += (away_odds * int(df_fixtures[config["columns"]["result"]].iloc[match_idx] == outcomes[2]) - 1) * away_odds * away_model_prob
        
    return returns

In [227]:
def best_strategy(config, df_fixtures, df_hda_odds, outcomes = ["H", "D", "A"]):

    thresholds = np.linspace(0, 8, 800)
    strategy_a_results = []
    strategy_b_results = []

    for threshold in thresholds:
        strategy_a_results.append(betting_strategy_a(config, df_fixtures, df_hda_odds, threshold))
        strategy_b_results.append(betting_strategy_b(config, df_fixtures, df_hda_odds, threshold))

    strategies = pd.DataFrame()
    strategies["Threshold"] = list(thresholds)
    strategies["Strategy_A"] = strategy_a_results
    strategies["Strategy_B"] = strategy_b_results

    best_a_idx = strategies["Strategy_A"].idxmax()
    best_a_threshold = strategies.loc[best_a_idx, "Threshold"]
    best_a_return = strategies.loc[best_a_idx, "Strategy_A"]

    best_b_idx = strategies["Strategy_B"].idxmax()
    best_b_threshold = strategies.loc[best_b_idx, "Threshold"]
    best_b_return = strategies.loc[best_b_idx, "Strategy_B"]

    if best_a_return >= best_b_return:
        return f"Strategy A: returns {best_a_return} at threshold {threshold}"
    else:
        return f"Strategy B: returns {best_b_return} at threshold {threshold}"


    


In [228]:
strategy_and_return = best_strategy(CONFIG, df_valid, df_hda_odds)

In [226]:
print(strategy_and_return)

Strategy A: returns 4.6499999999999995 at threshold 6.0


In [229]:
def make_test_data(config, df_fixtures):
    test_sets = []
    data_dir = Path(config["data_dir"])

    if config["split"]["method"] != "chronological_holdout":
        return ValueError("Unkown split method")
    else:
        for i in range(config["split"]["test_seasons"]):
            file_name = config["data_files"][len(config["data_files"]) + i - config["split"]["test_seasons"]]
            path = data_dir / file_name
            df_unfiltered = pd.read_csv(path)

            df =  df_unfiltered.loc[:, [config["columns"]["home_team"], config["columns"]["away_team"], 
                           config["columns"]["home_goals"], config["columns"]["away_goals"], 
                           config["columns"]["result"], config["columns"]["home_odds"], 
                           config["columns"]["draw_odds"], config["columns"]["away_odds"], config["columns"]["date"]]] 
            
            test_sets.insert(0, df)

        df_test_all = pd.concat(test_sets, ignore_index=True)

        home_col = config["columns"]["home_team"]
        away_col = config["columns"]["away_team"]

        training_teams = set(df_fixtures[home_col]) | set(df_fixtures[away_col])

        df_test = df_test_all[df_test_all[home_col].isin(training_teams) & df_test_all[away_col].isin(training_teams)].reset_index(drop=True)

        return df_test

In [230]:
df_test = make_test_data(CONFIG, df_train)

In [ ]:
best_rho = float(optimised_validation[0][0])
best_decay = float(optimised_validation[0][1])

best_phi = -np.log(1 - best_decay) / 365.25

opt_atk_def_coeffs = atk_def_coefficients(df_train, CONFIG, best_phi)

df_hda_odds_opt = hda_odds(CONFIG, df_test, opt_atk_def_coeffs[0], opt_atk_def_coeffs[1], best_rho)

test_log_loss = log_loss(CONFIG, df_test, df_hda_odds_opt)
print(test_log_loss)

True
Optimization terminated successfully
1.0524108167571449


In [236]:
test_brier = brier_score(CONFIG, df_test, optimised_validation[0][1], optimised_validation[0][0])

True
Optimization terminated successfully


In [237]:
print(test_brier)

0.634660950436836
